In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ClapProcessor, ClapModel
import librosa
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np

In [2]:
from datasets import load_from_disk

ds = load_from_disk("../01_load_dataset/musiccaps/dataset_audio")
len(ds)

# preliminary results
# framework direkt metric output edicek
# machine translation metrics, BLEU, ROUGE, METEOR, CIDEr, bertscore

Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

5310

In [3]:
# First split: separate test set (held out for final evaluation only)
ds_split = ds.train_test_split(test_size=0.1, seed=42)
ds_train_val = ds_split["train"]
ds_test = ds_split["test"]

# Second split: split train+val into train and validation sets
ds_train_val_split = ds_train_val.train_test_split(test_size=0.1, seed=42)
ds_train = ds_train_val_split["train"]
ds_val = ds_train_val_split["test"]

print(f"Train: {len(ds_train)}, Val: {len(ds_val)}, Test: {len(ds_test)}")

Train: 4301, Val: 478, Test: 531


In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

# CLAP (frozen)
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device)
clap.eval()
for p in clap.parameters():
    p.requires_grad = False

# GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2.eval()
for p in gpt2.parameters():
    p.requires_grad = False
    
# T5
# musicgen

/Users/aliozkaya/miniconda3/envs/audio-caption/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
D_AUDIO = clap.config.projection_dim      # e.g. 512
D_LM = gpt2.config.n_embd                 # 768
PREFIX_LEN = 8

In [6]:
projection = nn.Sequential(
    nn.Linear(D_AUDIO, D_LM * 2),      # 512 → 1536
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(D_LM * 2, D_LM * 4),     # 1536 → 3072
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(D_LM * 4, PREFIX_LEN * D_LM),  # 3072 → PREFIX_LEN * 768
).to(device)
projection.train()

Sequential(
  (0): Linear(in_features=512, out_features=1536, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=1536, out_features=3072, bias=True)
  (4): GELU(approximate='none')
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=3072, out_features=6144, bias=True)
)

In [7]:
optimizer = torch.optim.AdamW(
    list(projection.parameters()),
    lr=5e-4,
    weight_decay=0.1
)


In [8]:
def get_audio_embedding(sample):
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    if sr != 48000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=48000)

    inputs = clap_processor(
        audios=audio,
        sampling_rate=48000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        emb = clap.get_audio_features(**inputs)

    return emb  # (1, D_AUDIO)


In [9]:
def train_step(sample):
    audio_emb = get_audio_embedding(sample)     # (1, D_AUDIO)

    # Projection → prefix tokens
    prefix = projection(audio_emb)
    prefix = prefix.view(1, PREFIX_LEN, D_LM)

    # Tokenize caption
    tokens = tokenizer(
        sample["caption"],
        return_tensors="pt"
    ).to(device)

    input_ids = tokens.input_ids                 # (1, T)

    # Text embeddings
    text_embeds = gpt2.transformer.wte(input_ids)

    # Concatenate prefix + text
    inputs_embeds = torch.cat([prefix, text_embeds], dim=1)

    # Labels (ignore prefix)
    labels = input_ids.clone()
    ignore = torch.full((1, PREFIX_LEN), -100, device=device)
    labels = torch.cat([ignore, labels], dim=1)

    outputs = gpt2(
        inputs_embeds=inputs_embeds,
        labels=labels
    )

    return outputs.loss


In [10]:
from pathlib import Path

SAVE_DIR = Path("checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

In [ ]:
from tqdm import tqdm

best_val_loss = float("inf")

EPOCHS = 30

all_train_losses = []
all_val_losses = []
epoch_train_losses = []
epoch_val_losses = []

for epoch in tqdm(range(EPOCHS), desc="Training"):
    # ============ TRAINING ============
    projection.train()  # set to train mode
    total_train_loss = 0.0
    valid_samples = 0

    for i in tqdm(range(len(ds_train)), desc="Train", leave=False):
        try:
            optimizer.zero_grad()
            loss = train_step(ds_train[i])
            loss.backward()
            optimizer.step()
            
            all_train_losses.append(loss.item())
            total_train_loss += loss.item()
            valid_samples += 1
        except RuntimeError as e:
            if "No audio frames were decoded" in str(e):
                continue
            raise e

    avg_train_loss = total_train_loss / valid_samples
    epoch_train_losses.append(avg_train_loss)

    # ============ VALIDATION ============
    projection.eval()  # set to eval mode
    total_val_loss = 0.0
    val_samples = 0

    with torch.no_grad():  # no gradients needed for validation
        for i in tqdm(range(len(ds_val)), desc="Val", leave=False):
            try:
                loss = train_step(ds_val[i])  # same forward pass, no backward
                all_val_losses.append(loss.item())
                total_val_loss += loss.item()
                val_samples += 1
            except RuntimeError:
                continue

    avg_val_loss = total_val_loss / val_samples
    epoch_val_losses.append(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "model_state_dict": projection.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch + 1,
            "val_loss": best_val_loss,
        }, SAVE_DIR / "projection_best.pt")

    print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

Training:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
sample = ds_test[0]  # or any sample
audio_emb = get_audio_embedding(sample)
prefix = projection(audio_emb)
prefix = prefix.view(1, PREFIX_LEN, D_LM)

In [ ]:
generated = gpt2.generate(
    inputs_embeds=prefix,
    max_length=100,
    do_sample=True,          # Enable sampling
    temperature=0.8,         # Add randomness (0.7-1.0)
    top_k=50,                # Only sample from top 50 tokens
    top_p=0.9,               # Nucleus sampling
    repetition_penalty=1.2,  # Penalize repetition
    no_repeat_ngram_size=3,  # Prevent 3-gram repetition
)
print(tokenizer.decode(generated[0], skip_special_tokens=True))


In [ ]:
from IPython.display import Audio
Audio(ds[0]["audio"]["array"], rate=ds[0]["audio"]["sampling_rate"])

In [ ]:
plt.plot(all_losses)
plt.show()

In [ ]:
plt.plot(epoch_losses)
plt.show()

In [ ]:
# ============ STAGE 2: Unfreeze GPT-2 ============

# Unfreeze GPT-2
for p in gpt2.parameters():
    p.requires_grad = True
gpt2.train()
projection.train()

# New optimizer with different LRs
optimizer = torch.optim.AdamW([
    {"params": projection.parameters(), "lr": 1e-4},
    {"params": gpt2.parameters(), "lr": 5e-6},
], weight_decay=0.1)

EPOCHS_STAGE2 = 10
best_val_loss = float('inf')

for epoch in tqdm(range(EPOCHS_STAGE2), desc="Stage 2"):
    # ---- Training ----
    projection.train()
    gpt2.train()
    total_train_loss = 0.0
    train_samples = 0

    for i in tqdm(range(len(ds_train)), desc="Train", leave=False):
        try:
            optimizer.zero_grad()
            loss = train_step(ds_train[i])
            loss.backward()          # gradients for projection AND gpt2
            optimizer.step()         # updates projection at 1e-4, gpt2 at 5e-6
            
            total_train_loss += loss.item()
            train_samples += 1
        except RuntimeError:
            continue

    avg_train_loss = total_train_loss / train_samples

    # ---- Validation ----
    projection.eval()
    gpt2.eval()
    total_val_loss = 0.0
    val_samples = 0

    with torch.no_grad():
        for i in tqdm(range(len(ds_val)), desc="Val", leave=False):
            try:
                loss = train_step(ds_val[i])
                total_val_loss += loss.item()
                val_samples += 1
            except RuntimeError:
                continue

    avg_val_loss = total_val_loss / val_samples
    
    print(f"Epoch {epoch+1}: train={avg_train_loss:.4f}, val={avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'projection': projection.state_dict(),
            'gpt2': gpt2.state_dict(),
        }, 'best_model_stage2.pt')
        print(f"  ↳ Saved best model!")